# 03 - Portfolio Backtest

Backtest a custom portfolio allocation and compare against benchmarks.

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data import fetch_prices, compute_returns
from src.portfolio import (
    build_portfolio_returns, equity_curve, drawdown_series,
    rebalanced_portfolio_returns, compare_portfolios
)
from src.metrics import compute_all_metrics, metrics_table

## Define Portfolio Allocation

In [3]:
tickers = ["AAPL", "VWCE.MI", "BTC-USD", "^GSPC"]
start_date = "2018-01-01"
end_date = "2025-01-01"

prices = fetch_prices(tickers, start=start_date, end=end_date)
returns = compute_returns(prices)

weights = {
    "AAPL": 0.30,
    "VWCE.MI": 0.40,
    "BTC-USD": 0.10,
    "^GSPC": 0.20,
}

assert abs(sum(weights.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"
print(f"Portfolio: {weights}")

Portfolio: {'AAPL': 0.3, 'VWCE.MI': 0.4, 'BTC-USD': 0.1, '^GSPC': 0.2}


## Build Portfolio Returns

In [4]:
portfolio_returns = build_portfolio_returns(prices, weights)
rebalanced_returns = rebalanced_portfolio_returns(prices, weights, rebalance_freq='M')
benchmark_returns = returns['^GSPC']

print(f"Portfolio: {len(portfolio_returns)} days")
print(f"Rebalanced: {len(rebalanced_returns)} days")

Portfolio: 1231 days
Rebalanced: 1231 days


## Equity Curve

Compare: Buy-and-Hold vs Monthly-Rebalanced vs S&P 500 benchmark.

In [5]:
initial = 10000
portfolios = {
    'Buy & Hold': portfolio_returns,
    'Monthly Rebalanced': rebalanced_returns,
    'S&P 500': benchmark_returns,
}

curves = compare_portfolios(portfolios, initial_value=initial)

fig = go.Figure()
for col in curves.columns:
    fig.add_trace(go.Scatter(x=curves.index, y=curves[col], name=col, mode='lines'))

fig.update_layout(
    title=f'Portfolio Equity Curve (Initial: ${initial:,})',
    yaxis_title='Portfolio Value ($)',
    xaxis_title='Date',
    template='plotly_white',
    height=600,
)
fig.show()

## Drawdown Chart

In [6]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Equity Curve', 'Drawdown'),
                    vertical_spacing=0.08)

for col in curves.columns:
    fig.add_trace(go.Scatter(x=curves.index, y=curves[col], name=col, mode='lines'), row=1, col=1)

for name, ret in portfolios.items():
    dd = drawdown_series(ret) * 100
    fig.add_trace(go.Scatter(x=dd.index, y=dd, name=f'{name} DD', mode='lines'), row=2, col=1)

fig.update_layout(
    title='Portfolio Performance & Drawdown',
    template='plotly_white',
    height=700,
)
fig.update_yaxes(title_text='Value ($)', row=1, col=1)
fig.update_yaxes(title_text='Drawdown %', row=2, col=1)
fig.show()

## Portfolio Metrics Comparison

In [7]:
port_metrics = metrics_table(portfolios, benchmark_returns=benchmark_returns)
port_metrics_styled = port_metrics.style.format('{:.4f}').set_caption('Portfolio vs Benchmark Metrics')
port_metrics_styled

,CAGR,Annualized Return,Annualized Volatility,Sharpe Ratio,Sortino Ratio,Calmar Ratio,Omega Ratio,Max Drawdown,CVaR (95%),Skewness,Kurtosis,Best Day,Worst Day,Avg Daily Return,Beta,Treynor Ratio,Information Ratio,Alpha
Asset,,,,,,,,,,,,,,,,,,
Buy & Hold,0.2399,0.2399,0.2073,0.9644,1.1881,0.6250,1.1897,-0.3198,-0.0310,-0.7942,10.3033,0.0845,-0.1169,0.0009,0.8534,0.2342,0.9196,0.1250
Monthly Rebalanced,0.2362,0.2362,0.2072,0.9469,1.1644,0.5962,1.1861,-0.3291,-0.0310,-0.8169,10.2484,0.0845,-0.1165,0.0009,0.8514,0.2305,0.8829,0.1215
S&P 500,0.1278,0.1278,0.2150,0.4081,0.5005,0.2587,1.0965,-0.3392,-0.0325,-0.5180,12.9540,0.0938,-0.1198,0.0006,1.0000,0.0878,0.0000,0.0000


## Yearly Returns Heatmap

In [8]:
yearly = {}
for name, ret in portfolios.items():
    yearly[name] = ret.groupby(ret.index.year).apply(lambda x: (1 + x).prod() - 1)

yearly_df = pd.DataFrame(yearly)

fig = go.Figure(go.Heatmap(
    z=yearly_df.values * 100,
    x=yearly_df.columns,
    y=yearly_df.index,
    colorscale='RdYlGn',
    text=yearly_df.values.round(4) * 100,
    texttemplate='%{text:.1f}%',
))
fig.update_layout(
    title='Annual Returns (%)',
    template='plotly_white',
    height=max(400, len(yearly_df) * 40 + 200),
)
fig.show()

## Compare Different Allocations

In [9]:
allocations = {
    'Conservative': {'AAPL': 0.10, 'VWCE.MI': 0.70, 'BTC-USD': 0.0, '^GSPC': 0.20},
    'Balanced':     {'AAPL': 0.30, 'VWCE.MI': 0.40, 'BTC-USD': 0.10, '^GSPC': 0.20},
    'Aggressive':   {'AAPL': 0.40, 'VWCE.MI': 0.20, 'BTC-USD': 0.30, '^GSPC': 0.10},
}

alloc_returns = {}
for name, alloc in allocations.items():
    alloc_returns[name] = build_portfolio_returns(prices, alloc)
alloc_returns['S&P 500'] = benchmark_returns

alloc_table = metrics_table(alloc_returns, benchmark_returns=benchmark_returns)
alloc_table.style.format('{:.4f}').set_caption('Allocation Comparison')

,CAGR,Annualized Return,Annualized Volatility,Sharpe Ratio,Sortino Ratio,Calmar Ratio,Omega Ratio,Max Drawdown,CVaR (95%),Skewness,Kurtosis,Best Day,Worst Day,Avg Daily Return,Beta,Treynor Ratio,Information Ratio,Alpha
Asset,,,,,,,,,,,,,,,,,,
Conservative,0.1390,0.1390,0.1658,0.5971,0.7112,0.3038,1.1242,-0.3259,-0.0263,-0.6996,10.0234,0.0825,-0.0833,0.0006,0.6439,0.1537,0.0041,0.0425
Balanced,0.2399,0.2399,0.2073,0.9644,1.1881,0.6250,1.1897,-0.3198,-0.0310,-0.7942,10.3033,0.0845,-0.1169,0.0009,0.8534,0.2342,0.9196,0.1250
Aggressive,0.3823,0.3823,0.2936,1.1660,1.4973,0.8819,1.2225,-0.3882,-0.0420,-0.8431,9.8428,0.0921,-0.1761,0.0015,1.0059,0.3403,1.1285,0.2541
S&P 500,0.1278,0.1278,0.2150,0.4081,0.5005,0.2587,1.0965,-0.3392,-0.0325,-0.5180,12.9540,0.0938,-0.1198,0.0006,1.0000,0.0878,0.0000,0.0000


In [10]:
alloc_curves = compare_portfolios(alloc_returns, initial_value=10000)

fig = go.Figure()
for col in alloc_curves.columns:
    fig.add_trace(go.Scatter(x=alloc_curves.index, y=alloc_curves[col], name=col, mode='lines'))
fig.update_layout(
    title='Allocation Comparison (Initial: $10,000)',
    yaxis_title='Portfolio Value ($)',
    xaxis_title='Date',
    template='plotly_white',
    height=600,
)
fig.show()